In [2]:
!pip -q install pyspark


In [3]:
from google.colab import drive
import os
drive.mount('/content/drive')

LANDING = "/content/drive/MyDrive/Parcial Mineria de Datos II/Proyecto/cloud_provider_challenge_dataset_v1/datalake/landing"

for f in sorted(os.listdir(LANDING)):
    print(f)
print(len(os.listdir(f"{LANDING}/usage_events_stream")), "archivos de eventos")

Mounted at /content/drive
billing_monthly.csv
customers_orgs.csv
marketing_touches.csv
nps_surveys.csv
resources.csv
support_tickets.csv
usage_events_stream
users.csv
120 archivos de eventos


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .appName("cloud-provider-exploracion")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
print("Spark", spark.version)

Spark 4.0.4


In [5]:
def perfil(df, nombre):
    print(f"\n{'='*20} {nombre} {'='*20}")
    print("Filas:", df.count(), "| Columnas:", len(df.columns))
    df.printSchema()
    print("Nulos por columna:")
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
               for c in df.columns]).show(vertical=True)
    df.show(3, truncate=False)

In [6]:
csvs = ["customers_orgs", "users", "resources", "support_tickets",
        "marketing_touches", "nps_surveys", "billing_monthly"]
dfs = {}
for nombre in csvs:
    dfs[nombre] = (spark.read
                   .option("header", True)
                   .option("inferSchema", True)
                   .csv(f"{LANDING}/{nombre}.csv"))
    perfil(dfs[nombre], nombre)


==================== customers_orgs ====================
Filas: 80 | Columnas: 11
root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)

Nulos por columna:
-RECORD 0---------------
 org_id           | 0   
 org_name         | 0   
 industry         | 0   
 hq_region        | 0   
 plan_tier        | 0   
 is_enterprise    | 0   
 signup_date      | 0   
 sales_rep        | 0   
 lifecycle_stage  | 0   
 marketing_source | 0   
 nps_score        | 11  

+------------+-------------+---------+----------+---------+-------------+-----------+---------+---------------+----------------

In [7]:
eventos = spark.read.json(f"{LANDING}/usage_events_stream/*.jsonl")
perfil(eventos, "usage_events")
eventos.groupBy("schema_version").count().show()


==================== usage_events ====================
Filas: 43200 | Columnas: 13
root
 |-- carbon_kg: double (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- event_id: string (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- metric: string (nullable = true)
 |-- org_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- schema_version: long (nullable = true)
 |-- service: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- value: string (nullable = true)

Nulos por columna:
-RECORD 0-------------------
 carbon_kg          | 10800 
 cost_usd_increment | 0     
 event_id           | 0     
 genai_tokens       | 40068 
 metric             | 0     
 org_id             | 0     
 region             | 0     
 resource_id        | 0     
 schema_version     | 0     
 service            | 0     
 timestamp          | 0     
 unit           

In [8]:
ev = (eventos
      .withColumn("ts", F.expr("try_to_timestamp(timestamp)"))
      .withColumn("value_num", F.expr("try_cast(value AS DOUBLE)")))

print("event_id duplicados:", ev.groupBy("event_id").count().filter("count > 1").count())
print("timestamps que no se pudieron convertir:", ev.filter("ts IS NULL").count())
ev.select(F.min("ts").alias("desde"), F.max("ts").alias("hasta")).show()

neg = ev.filter("cost_usd_increment < 0")
neg.select(F.min("cost_usd_increment").alias("mas_negativo"),
           F.max("cost_usd_increment").alias("menos_negativo")).show()
print("Negativos dentro de la tolerancia (>= -0.01):",
      neg.filter("cost_usd_increment >= -0.01").count())
neg.groupBy("service").count().orderBy(F.desc("count")).show()

print("Costos negativos:", ev.filter("cost_usd_increment < 0").count())
ev.select(
    F.expr("percentile(cost_usd_increment, 0.5)").alias("mediana"),
    F.expr("percentile(cost_usd_increment, 0.99)").alias("p99"),
    F.max("cost_usd_increment").alias("maximo")).show()

print("value no numérico:", ev.filter("value IS NOT NULL AND value_num IS NULL").count())
ev.filter("value IS NOT NULL AND value_num IS NULL").select("value").distinct().show(10)

event_id duplicados: 0
timestamps que no se pudieron convertir: 0
+-------------------+-------------------+
|              desde|              hasta|
+-------------------+-------------------+
|2025-07-03 00:02:00|2025-08-31 23:58:00|
+-------------------+-------------------+

+------------+--------------+
|mas_negativo|menos_negativo|
+------------+--------------+
|   -154.4608|       -0.0029|
+------------+--------------+

Negativos dentro de la tolerancia (>= -0.01): 5
+----------+-----+
|   service|count|
+----------+-----+
|   compute|   56|
|  database|   44|
|   storage|   44|
|networking|   32|
|     genai|   21|
| analytics|   19|
+----------+-----+

Costos negativos: 216
+-------+------------------+--------+
|mediana|               p99|  maximo|
+-------+------------------+--------+
| 1.0044|16.717518000000005|317.4308|
+-------+------------------+--------+

value no numérico: 0
+-----+
|value|
+-----+
+-----+



In [9]:
ev.groupBy("region").count().orderBy(F.desc("count")).show(truncate=False)
ev.groupBy("service").count().orderBy(F.desc("count")).show(truncate=False)
dfs["customers_orgs"].groupBy("hq_region").count().show(truncate=False)

dfs["billing_monthly"].groupBy("currency").agg(
    F.count("*").alias("facturas"),
    F.min("exchange_rate_to_usd").alias("fx_min"),
    F.max("exchange_rate_to_usd").alias("fx_max")).show()

+------------+-----+
|region      |count|
+------------+-----+
|ap-northeast|7773 |
|us-east     |7566 |
|us-west     |6384 |
|sa-east     |6270 |
|ap-south    |5943 |
|eu-west     |4854 |
|eu-central  |4410 |
+------------+-----+

+----------+-----+
|service   |count|
+----------+-----+
|compute   |12498|
|storage   |7614 |
|database  |7419 |
|networking|6921 |
|analytics |4590 |
|genai     |4158 |
+----------+-----+

+------------+-----+
|hq_region   |count|
+------------+-----+
|us-west     |7    |
|eu-central  |16   |
|us-east     |14   |
|ap-northeast|11   |
|sa-east     |10   |
|ap-south    |15   |
|eu-west     |7    |
+------------+-----+

+--------+--------+-------+-------+
|currency|facturas| fx_min| fx_max|
+--------+--------+-------+-------+
|     ARS|      51|0.00133|0.00162|
|     USD|     160|0.85463|1.11791|
|     EUR|      29| 0.9981|1.19808|
+--------+--------+-------+-------+



In [10]:
for nombre in ["customers_orgs", "nps_surveys"]:
    print(f"\n=== {nombre} ===")
    dfs[nombre].select(
        F.min("nps_score").alias("min"),
        F.max("nps_score").alias("max"),
        F.expr("percentile(nps_score, 0.5)").alias("mediana")).show()
    dfs[nombre].groupBy("nps_score").count().orderBy("nps_score").show(50)


=== customers_orgs ===
+-----+-----+-------+
|  min|  max|mediana|
+-----+-----+-------+
|-38.0|101.0|   14.0|
+-----+-----+-------+

+---------+-----+
|nps_score|count|
+---------+-----+
|     NULL|   11|
|    -38.0|    1|
|    -33.0|    1|
|    -28.0|    1|
|    -17.0|    1|
|    -13.0|    1|
|     -8.0|    1|
|     -7.0|    2|
|     -6.0|    1|
|     -4.0|    2|
|     -3.0|    1|
|     -2.0|    1|
|     -1.0|    3|
|      0.0|    1|
|      1.0|    1|
|      2.0|    1|
|      3.0|    1|
|      4.0|    4|
|      9.0|    1|
|     10.0|    1|
|     11.0|    4|
|     12.0|    3|
|     14.0|    4|
|     17.0|    2|
|     19.0|    1|
|     20.0|    1|
|     21.0|    2|
|     22.0|    1|
|     25.0|    3|
|     26.0|    1|
|     27.0|    1|
|     30.0|    1|
|     34.0|    2|
|     36.0|    1|
|     37.0|    1|
|     41.0|    3|
|     44.0|    2|
|     45.0|    1|
|     50.0|    2|
|     51.0|    1|
|     53.0|    1|
|     62.0|    1|
|     65.0|    1|
|     79.0|    1|
|     81.0|    1|
|